In [33]:
from ollama import chat
from pydantic import BaseModel, Field
from typing import List, Optional

import json

In [34]:
folder_name = "cortex_Booeshaghi2021"
img_fn = "figure.jpg"
fn = f"../../data/{folder_name}/{img_fn}"

In [35]:
# Restrictive class object does not allow for missing fields
class MarkerStrict(BaseModel):
    marker_name: str = Field(
        ..., 
        description="The name of the marker (can be a gene or isoform) that is associated with a specific cell type."
    )
    cell_type_name: str = Field(
        ..., 
        description="The name of the cell type for which this gene or isoform serves as a marker."
    )
    tissue_source: str = Field(
        ..., 
        description="The tissue/location/origin from which this cell type was identified"
    )
    species: str = Field(
        ..., 
        description="The species from which this marker was identified, e.g., 'Homo sapiens', 'Mus musculus'."
    )

class MarkerListStrict(BaseModel):
    genes: List[MarkerStrict] 

In [36]:
# Currently, we assume that the figure is an umbrella plot of marker genes.

system_prompt = """
You are an expert in genomics analyzing scientific literature to extract marker isoforms for different cell types. 

Your goal is to identify and structure marker gene data from the given text. For each marker isoform mentioned, extract:
- The **isoform name** (marker_gene_name).
- The **cell type** it is associated with (cell_type_name).
- The **tissue source** where this marker gene was identified (tissue_source).
- The **species** from which the marker gene was studied (species).

If any information is missing or ambiguous, provide the most specific details available. If none of the information is available, the field can be set to null.

The data must be extracted as written in the text, without any modifications.

Return the results in **structured JSON format** with the following schema:
{
    "genes": [
        {
            "marker_gene_name": "GeneX",
            "cell_type_name": "Neuron",
            "tissue_source": "Brain",
            "species": "Homo sapiens"
        },
        ...
    ]
}
"""

user_content = "Given this umbrella plot figure, return a list of markers and their cell types."

In [37]:
def extract_genes(user_content, system_prompt, data_model, model="llava"):
    response = chat(
        messages=[
            {'role': 'system',
             'content': system_prompt},
            {
                'role': 'user',
                'content': "Spatial isoform atlas of the MOp. The scatter plots (left) show the locations of cells (black dots) in the indicated subclass within a single representative slice of the mouse MOp as assayed by MERFISH. Right, each column corresponds to a marker gene in the MERFISH dataset and each row corresponds to a subclass (number of cells in parentheses) in which one isoform (labelled on the diagonal) was differential in the SMART-seq dataset for that subclass. This spatial isoform inference links isoform expression from the SMART-seq data with the physical locations of cells expressing that isoform from the MERFISH data. The normalized gene expression values are plotted for each subclass–gene pair in TPM. The white circles in the violin plots represent the mean and the white bars represent the s.d. Please identify the marker isoforms (marked columns) for each row (aka subclass)",
                'images': [fn]
            }
        ],
        model = model,
        options={'temperature': 0},  # Make responses more deterministic

    )
    return response['message']['content']


In [38]:
genes = extract_genes(user_content, system_prompt, MarkerListStrict)
print(genes)

 Based on the image provided, here are the marker isoforms identified for each cell type:

- Neuron:
  - GeneX

The tissue source for all markers is "Brain," and the species is "Homo sapiens." The cell types are not explicitly labeled in the image, but based on the context provided, it seems that the different rows represent different subclasses of cells within a single representative slice of the mouse MOp.

Please note that the image does not provide enough information to determine if there are additional marker isoforms for each subclass or if this is a complete list. The image also includes a legend explaining the color coding and symbols used in the heatmap, but it does not provide specific details about the markers themselves. 
